#### Optimizer r12: single-scenario optimizer with summary tables and plots.

In [ ]:

from pathlib import Path
import copy
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.demand_linear_r02 import TaylorLinearDemandModel
from src.costs import CostModel
from src.benefits_linear_r03 import TaylorLinearBenefitModel

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

try:
    import gurobipy as gp
    from gurobipy import GRB, quicksum
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        'gurobipy is not installed in this environment. Run this notebook in your local project environment.'
    ) from e

from IPython.display import display


#### Controls

In [ ]:
ROUTES_OF_INTEREST = ['10', '111', '48', '7']
TIME_BLOCKS = ['Early AM', 'AM Peak', 'Midday', 'PM Peak', 'Evening', 'Night']

OPTIMIZATION_SETUP = {
    'output_flag': 0,
}

FLEET_SCENARIO_SETUP = {
    'mode': 'expanded',   # 'baseline' or 'expanded'
}

BASE_SCENARIO = {
    'max_change_by_route_type': {
        'Frequent': 4,
        'Local': 2,
        'Connexion': 2,
    },
    'expansion_factor': 1.1,
    'fleet_caps': None,   # use None to let FLEET_SCENARIO_SETUP choose baseline or expanded caps automatically
    'budget_total': 50000.0,
    'demand_elasticity': 0.5,
    'W_driver_multiplier': 1.0,
    'P_fuel_multiplier': 1.0,
    'fuel_consumption_multiplier': 1.0,
    'P_maintenance_multiplier': 1.0,
    'W_passenger_multiplier': 1.0,
    'F_wage_multiplier': 1.0,
    'I_GHG_car_multiplier': 1.0,
    'L_avg_trip_multiplier': 1.0,
    'V_km_saved_multiplier': 1.0,
    'SCC_multiplier': 1.0,
}

TABLE_SETUP = {
    'route_summary_time_block': 'all',   # 'all' or a time block name
    'top_n_routes': 15,
    'sort_route_summary_by': 'benefit_total_delta_total',  # any route-summary column
    'ascending': False,
}

GRAPH_SETUP = {
    'time_block': 'all',   # use 'all' for grouped bars by time block, or a single time block name for a simpler chart
    'value_col': 'n_new',  # 'n_new' or 'delta_n'
    'sort_routes_numeric': True,
    'show_baseline_ticks': True,
    'show_delta_labels': True,
    'show_legend_baseline': True,
    'color_bars_by_delta': False,  # mainly for single-time-block charts; grouped time-block charts use bar colors for time blocks
}


#### Load and clean data

In [ ]:
#helper to find the first existing file path from a list of candidates, or raise an error if none are found
def first_existing_path(candidates):
    path = next((Path(p) for p in candidates if Path(p).exists()), None)
    if path is None:
        raise FileNotFoundError(f'Could not find any of: {candidates}')
    return path


def load_and_clean_data():
    route_data_path = first_existing_path([
        'data/route_timeblock_v03.csv',
        '/mnt/data/route_timeblock_v03.csv',
        'data/route_timeblock_v02.csv',
        'data/route_timeblock_subset_v02.csv',
        'data/route_timeblock_subset_v01.csv',
        '/mnt/data/route_timeblock_subset_v01.csv',
    ])

    raw_df = pd.read_csv(route_data_path)

    col_map = {
        'Route (r)': 'route',
        'Time Block (t)': 'time_block',
        'Route type': 'route_type',
        'Old ridership (x_old,r,t)': 'x_old',
        'Trip length km (L_r)': 'L_r',
        'Average length hr (T_r,t)': 'T_rt',
        'Old frequency of buses (f_old,r,t)': 'f_old',
        'Old number of buses (n_old,r,t) - continuous': 'n_old_cont',
        'Old number of buses (n_old,r,t) - discrete': 'n_old',
        'Time block hr (H_block,t)': 'H_block',
        'Driver horuly wage rate $/hr (W_driver)': 'W_driver',
        'Price of fuel $/litre (P_fuel)': 'P_fuel',
        'Fuel consumption (FC)': 'fuel_consumption',
        'Maintenace cost $/km (P_maintenance)': 'P_maintenance',
        'Passenger hourly wage rate $/hr (W_passenger)': 'W_passenger',
        'VTTS fraction of passenger wage (F_wage)': 'F_wage',
        'Emission intensity of a car kgCO2eq/KPT (I_GHG,car)': 'I_GHG_car',
        'Average Length of passenger trip (L_avg_trip)': 'L_avg_trip',
        'Social cost of carbon $/tonne CO2eq (SCC)': 'SCC',
        ' Vehicle-miles saved per 1 km  of passenger km travelled (V_km,saved)': 'V_km_saved',
    }

    df = raw_df.rename(columns=col_map).copy()

    required_cols = [
        'route', 'time_block', 'route_type',
        'x_old', 'L_r', 'T_rt', 'f_old',
        'n_old_cont', 'n_old',
        'H_block', 'W_driver', 'P_fuel',
        'fuel_consumption', 'P_maintenance',
        'W_passenger', 'F_wage',
        'I_GHG_car', 'L_avg_trip', 'V_km_saved', 'SCC'
    ]

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f'Missing expected columns: {missing}')

    df['route'] = df['route'].astype(str).str.strip()
    df['time_block'] = df['time_block'].astype(str).str.strip()
    df['route_type'] = df['route_type'].astype(str).str.strip()

    cancelled_mask = df['route_type'].str.lower().eq('cancelled')
    df = df.loc[~cancelled_mask].copy()

    numeric_cols = [
        'x_old', 'L_r', 'T_rt', 'f_old',
        'n_old_cont', 'n_old',
        'H_block', 'W_driver', 'P_fuel',
        'fuel_consumption', 'P_maintenance',
        'W_passenger', 'F_wage',
        'I_GHG_car', 'L_avg_trip', 'V_km_saved', 'SCC'
    ]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    df['n_old'] = df['n_old'].round().astype(int)
    df['time_block'] = pd.Categorical(df['time_block'], categories=TIME_BLOCKS, ordered=True)
    df = df.sort_values(['route', 'time_block']).reset_index(drop=True)
    df['key'] = list(zip(df['route'], df['time_block'].astype(str)))

    return df, route_data_path


df, route_data_path = load_and_clean_data()
print(f'Using route-time data: {route_data_path}')
df.head()


baseline_fleet_caps = (
    df.groupby('time_block', observed=True)['n_old']
    .sum()
    .astype(int)
    .to_dict()
)

BASE_SCENARIO_ACTIVE = copy.deepcopy(BASE_SCENARIO)

# choose expansion factor from scenario (this is the key change)
expansion = BASE_SCENARIO_ACTIVE['expansion_factor']

expanded_fleet_caps = {
    t: int(math.ceil(baseline_fleet_caps[t] * expansion))
    for t in TIME_BLOCKS
}

if FLEET_SCENARIO_SETUP['mode'] == 'baseline':
    BASE_SCENARIO_ACTIVE['fleet_caps'] = baseline_fleet_caps.copy()

elif FLEET_SCENARIO_SETUP['mode'] == 'expanded':
    BASE_SCENARIO_ACTIVE['fleet_caps'] = expanded_fleet_caps.copy()

else:
    raise ValueError("FLEET_SCENARIO_SETUP['mode'] must be 'baseline' or 'expanded'")

print(f"Using fleet scenario: {FLEET_SCENARIO_SETUP['mode']}")
print(f"Fleet caps: {BASE_SCENARIO_ACTIVE['fleet_caps']}")


#### Build model inputs

In [ ]:
# helper to build all optimization inputs from the scenario and dataframe,
# including the first-order Taylor coefficients for demand and benefits.
def build_inputs_from_scenario(df, scenario):
    R = sorted(df['route'].unique().tolist())
    T = list(df['time_block'].cat.categories)
    RT = [(r, t) for r in R for t in T]

    missing = set(RT) - set(df['key'])
    if missing:
        raise ValueError(f'Missing route-time combinations: {sorted(missing)}')

    route_types_in_df = sorted(df['route_type'].dropna().unique())
    missing_route_types = sorted(set(route_types_in_df) - set(scenario['max_change_by_route_type'].keys()))
    if missing_route_types:
        raise KeyError(f'Missing max-change caps for route types: {missing_route_types}')

    df2 = df.copy()

    df2['W_driver_eff'] = df2['W_driver'] * scenario['W_driver_multiplier']
    df2['P_fuel_eff'] = df2['P_fuel'] * scenario['P_fuel_multiplier']
    df2['fuel_consumption_eff'] = df2['fuel_consumption'] * scenario['fuel_consumption_multiplier']
    df2['P_maintenance_eff'] = df2['P_maintenance'] * scenario['P_maintenance_multiplier']
    df2['W_passenger_eff'] = df2['W_passenger'] * scenario['W_passenger_multiplier']
    df2['F_wage_eff'] = df2['F_wage'] * scenario['F_wage_multiplier']
    df2['I_GHG_car_eff'] = df2['I_GHG_car'] * scenario['I_GHG_car_multiplier']
    df2['L_avg_trip_eff'] = df2['L_avg_trip'] * scenario['L_avg_trip_multiplier']
    df2['V_km_saved_eff'] = df2['V_km_saved'] * scenario['V_km_saved_multiplier']
    df2['SCC_eff'] = df2['SCC'] * scenario['SCC_multiplier']

    x_old = df2.set_index('key')['x_old'].to_dict()
    n_old = df2.set_index('key')['n_old'].astype(int).to_dict()
    T_rt = df2.set_index('key')['T_rt'].to_dict()
    route_type = df2.set_index('key')['route_type'].to_dict()
    V_km_saved = df2.set_index('key')['V_km_saved_eff'].to_dict()

    L_r = df2.groupby('route')['L_r'].first().to_dict()
    H_block = df2.groupby('time_block')['H_block'].first().to_dict()

    W_driver = float(df2['W_driver_eff'].iloc[0])
    P_fuel = float(df2['P_fuel_eff'].iloc[0])
    P_maintenance = float(df2['P_maintenance_eff'].iloc[0])
    fuel_consumption = float(df2['fuel_consumption_eff'].mean())

    alpha = {}
    beta_time = {}
    beta_emissions = {}
    n_min = {}
    n_max = {}
    frozen_zero_keys = []
    RT_opt = []

    eps = float(scenario['demand_elasticity'])

    for _, row in df2.iterrows():
        r = row['route']
        t = str(row['time_block'])
        k = (r, t)

        x0 = float(row['x_old'])
        T0 = float(row['T_rt'])
        n0 = int(row['n_old'])
        rt_type = row['route_type']

        if pd.isna(T0) or T0 < 0:
            raise ValueError(f'Invalid T_rt for key {k}: {T0}')

        if n0 <= 0:
            alpha[k] = 0.0
            beta_time[k] = 0.0
            beta_emissions[k] = 0.0
            n_min[k] = 0
            n_max[k] = 0
            frozen_zero_keys.append(k)
            continue

        if T0 <= 0:
            raise ValueError(f'T_rt must be positive when n_old > 0 for key {k}')

        delta_cap = int(scenario['max_change_by_route_type'][rt_type])
        n_min[k] = max(0, n0 - delta_cap)
        n_max[k] = n0 + delta_cap

        # Demand Taylor coefficient:
        # x(n) = x_old * (n / n_old)^eps
        # alpha = dx/dn |_(n_old) = eps * x_old / n_old
        alpha[k] = eps * x0 / n0

        # Time-benefit Taylor coefficient:
        # ΔB_time(n) = (T_rt/2) * (1/n_old - 1/n) * x_old * (n/n_old)^eps * W_passenger * F_wage
        # beta_time = d(ΔB_time)/dn |_(n_old) = x_old * W_passenger * F_wage * T_rt / (2 * n_old^2)
        beta_time[k] = (
            x0
            * float(row['W_passenger_eff'])
            * float(row['F_wage_eff'])
            * T0
            / (2.0 * (n0 ** 2))
        )

        # Emissions-benefit Taylor coefficient before V_km_saved:
        # ΔB_em(n) = K * V_km_saved * (x(n) - x_old)
        # where K = L_avg_trip * I_GHG_car * SCC / 1000
        # beta_emissions = d(ΔB_em)/dn |_(n_old) without V_km_saved;
        # the model applies V_km_saved separately so the new data column is explicit.
        beta_emissions[k] = (
            alpha[k]
            * float(row['L_avg_trip_eff'])
            * float(row['I_GHG_car_eff'])
            * float(row['SCC_eff'])
            / 1000.0
        )

        RT_opt.append(k)

    return {
        'R': R,
        'T': T,
        'RT': RT,
        'RT_opt': RT_opt,
        'x_old': x_old,
        'n_old': n_old,
        'T_rt': T_rt,
        'route_type': route_type,
        'L_r': L_r,
        'H_block': H_block,
        'W_driver': W_driver,
        'P_fuel': P_fuel,
        'fuel_consumption': fuel_consumption,
        'P_maintenance': P_maintenance,
        'alpha': alpha,
        'beta_time': beta_time,
        'beta_emissions': beta_emissions,
        'V_km_saved': V_km_saved,
        'n_min': n_min,
        'n_max': n_max,
        'fleet_caps': scenario['fleet_caps'],
        'budget_total': scenario['budget_total'],
        'frozen_zero_keys': frozen_zero_keys,
    }


#### Solve one scenario

In [ ]:

#consstruct optimization problem based on formulation from /src and solve it, returning the solution and summary statistics in a structured format
def solve_scenario(df, scenario, model_name='oc_transpo_taylor_linearized_r10', output_flag=0):
    data = build_inputs_from_scenario(df, scenario)




    #-------------------------------------------------------------------------------------------------------
    #
    #
    # Model / Obj Function

    demand_model = TaylorLinearDemandModel(
        alpha=data['alpha'],
        n_old=data['n_old'],
        x_old=data['x_old'],
    )

    cost_model = CostModel(
        H_block=data['H_block'],
        W_driver=data['W_driver'],
        L_r=data['L_r'],
        T_rt=data['T_rt'],
        P_fuel=data['P_fuel'],
        fuel_consumption=data['fuel_consumption'],
        P_maintenance=data['P_maintenance'],
        n_old=data['n_old'],
    )

    benefit_model = TaylorLinearBenefitModel(
        beta_time=data['beta_time'],
        beta_emissions=data['beta_emissions'],
        n_old=data['n_old'],
        V_km_saved=data['V_km_saved'],
    )

    # Build and solve optimization model based on the data and models
    m = gp.Model(model_name)
    m.Params.OutputFlag = output_flag

    # Decision variables: n_new[r,t] for each route-time in RT_opt
    n_new = m.addVars(
        data['RT_opt'],
        vtype=GRB.INTEGER,
        lb={k: data['n_min'][k] for k in data['RT_opt']},
        ub={k: data['n_max'][k] for k in data['RT_opt']},
        name='n_new',
    )
 
    # Objective: minimize total cost minus total benefit across all route-times in RT_opt
    objective = quicksum(
        cost_model.total(r, t, n_new[r, t]) - benefit_model.benefit_total(r, t, n_new[r, t])
        for r, t in data['RT_opt']
    )
    m.setObjective(objective, GRB.MINIMIZE)


    #-------------------------------------------------------------------------------------------------------
    #
    #
    # Constraints: 
    fleet_constraints = {
        t: m.addConstr(
            quicksum(n_new[r, tb] for (r, tb) in data['RT_opt'] if tb == t) <= data['fleet_caps'][t],
            name=f'fleet[{t}]'
        )
        for t in data['T']
    }

    m.addConstr(
        quicksum(cost_model.total(r, t, n_new[r, t]) for r, t in data['RT_opt']) <= data['budget_total'],
        name='budget_total_operational'
    )

    m.optimize()

    if m.Status != GRB.OPTIMAL:
        raise RuntimeError(f'Model did not solve to optimality. Status = {m.Status}')

    rows = []

    for r, t in data['RT']:
        active = (r, t) in data['RT_opt']

        if active:
            n_star = int(round(n_new[r, t].X))
            x_new = demand_model.ridership(r, t, n_star)
            delta_x = demand_model.delta_ridership(r, t, n_star)
            cost_delta = cost_model.total(r, t, n_star)
            benefit_time_delta = benefit_model.benefit_time(r, t, n_star)
            benefit_emissions_delta = benefit_model.benefit_emissions(r, t, n_star)
            benefit_total_delta = benefit_model.benefit_total(r, t, n_star)
            net_contrib = cost_delta - benefit_total_delta
        else:
            n_star = 0
            x_new = data['x_old'][(r, t)]
            delta_x = 0.0
            cost_delta = 0.0
            benefit_time_delta = 0.0
            benefit_emissions_delta = 0.0
            benefit_total_delta = 0.0
            net_contrib = 0.0

        rows.append({
            'route': r,
            'time_block': t,
            'route_type': data['route_type'][(r, t)],
            'active_in_model': active,
            'n_old': data['n_old'][(r, t)],
            'n_new': n_star,
            'delta_n': n_star - data['n_old'][(r, t)],
            'x_old': data['x_old'][(r, t)],
            'x_new_linear': x_new,
            'V_km_saved': data['V_km_saved'][(r, t)],
            'delta_ridership': delta_x,
            'operating_cost_delta': cost_delta,
            'benefit_time_delta': benefit_time_delta,
            'benefit_emissions_delta': benefit_emissions_delta,
            'benefit_total_delta': benefit_total_delta,
            'net_objective_contribution': net_contrib,
            'at_lower_bound': n_star == data['n_min'][(r, t)],
            'at_upper_bound': n_star == data['n_max'][(r, t)],
        })

    solution_df = pd.DataFrame(rows)

    summary = {
        'objective_value': m.ObjVal,
        'budget_total': data['budget_total'],
        'budget_used': solution_df['operating_cost_delta'].sum(),
        'budget_slack': data['budget_total'] - solution_df['operating_cost_delta'].sum(),
        'total_delta_ridership': solution_df['delta_ridership'].sum(),
        'total_cost_delta': solution_df['operating_cost_delta'].sum(),
        'total_time_benefit_delta': solution_df['benefit_time_delta'].sum(),
        'total_emissions_benefit_delta': solution_df['benefit_emissions_delta'].sum(),
        'total_benefit_delta': solution_df['benefit_total_delta'].sum(),
        'vars_at_lower_bound': int(solution_df['at_lower_bound'].sum()),
        'vars_at_upper_bound': int(solution_df['at_upper_bound'].sum()),
        'runtime_seconds': m.Runtime,
        'node_count': m.NodeCount,
        'mip_gap': m.MIPGap,
        'solver_status': m.Status,
        'sol_count': m.SolCount,
        'iter_count': m.IterCount,
        'num_vars': m.NumVars,
        'num_constrs': m.NumConstrs,    
    }

    for t in data['T']:
        used = int(solution_df.loc[solution_df['time_block'] == t, 'n_new'].sum())
        cap = data['fleet_caps'][t]
        summary[f'fleet_used[{t}]'] = used
        summary[f'fleet_cap[{t}]'] = cap
        summary[f'fleet_slack[{t}]'] = cap - used

    return {
        'model': m,
        'solution_df': solution_df,
        'summary': pd.Series(summary),
        'scenario': copy.deepcopy(scenario),
    }


#### Summary helpers

In [ ]:

def make_route_summary(solution_df, time_block='all'):
    df_plot = solution_df.copy()

    if time_block != 'all':
        if time_block not in df_plot['time_block'].unique():
            raise ValueError(f"Unknown time_block: {time_block}")
        df_plot = df_plot[df_plot['time_block'] == time_block].copy()

    route_summary = (
        df_plot.groupby('route', as_index=False)
        .agg(
            n_old_total=('n_old', 'sum'),
            n_new_total=('n_new', 'sum'),
            delta_n_total=('delta_n', 'sum'),
            x_old_total=('x_old', 'sum'),
            x_new_linear_total=('x_new_linear', 'sum'),
            delta_ridership_total=('delta_ridership', 'sum'),
            operating_cost_delta_total=('operating_cost_delta', 'sum'),
            benefit_time_delta_total=('benefit_time_delta', 'sum'),
            benefit_emissions_delta_total=('benefit_emissions_delta', 'sum'),
            benefit_total_delta_total=('benefit_total_delta', 'sum'),
            net_objective_contribution_total=('net_objective_contribution', 'sum'),
        )
    )

    return route_summary


def make_timeblock_summary(solution_df):
    return (
        solution_df.groupby('time_block', as_index=False)
        .agg(
            n_old_total=('n_old', 'sum'),
            n_new_total=('n_new', 'sum'),
            delta_n_total=('delta_n', 'sum'),
            delta_ridership_total=('delta_ridership', 'sum'),
            operating_cost_delta_total=('operating_cost_delta', 'sum'),
            benefit_time_delta_total=('benefit_time_delta', 'sum'),
            benefit_emissions_delta_total=('benefit_emissions_delta', 'sum'),
            benefit_total_delta_total=('benefit_total_delta', 'sum'),
            net_objective_contribution_total=('net_objective_contribution', 'sum'),
        )
    )


def make_selected_route_detail(solution_df, routes_of_interest):
    return (
        solution_df[solution_df['route'].isin(routes_of_interest)]
        .sort_values(['route', 'time_block'])
        .reset_index(drop=True)
    )


def aggregate_for_plot(solution_df, time_block='all', value_col='n_new'):
    if value_col not in ['n_new', 'delta_n']:
        raise ValueError("value_col must be 'n_new' or 'delta_n'")

    df_plot = solution_df.copy()

    if time_block != 'all':
        if time_block not in df_plot['time_block'].unique():
            raise ValueError(f"Unknown time_block: {time_block}")
        df_plot = df_plot[df_plot['time_block'] == time_block].copy()

    grouped = (
        df_plot.groupby('route', as_index=False)[['n_old', 'n_new', 'delta_n']]
        .sum()
    )

    return grouped


def sort_route_index(index_vals):
    return sorted(
        index_vals,
        key=lambda x: int(x) if str(x).isdigit() else str(x)
    )


def make_plot_frame(
    solution_df,
    time_block='all',
    value_col='n_new',
    route_filter=None,
    sort_routes_numeric=True,
):
    grouped = aggregate_for_plot(
        solution_df=solution_df,
        time_block=time_block,
        value_col=value_col,
    )

    if route_filter is not None:
        grouped = grouped[grouped['route'].isin(route_filter)].copy()

    if route_filter is not None:
        target_index = route_filter
    elif sort_routes_numeric:
        target_index = sort_route_index(grouped['route'].tolist())
    else:
        target_index = grouped['route'].tolist()

    grouped = grouped.set_index('route').reindex(target_index).reset_index()
    grouped['plot_value'] = grouped[value_col]

    return grouped


def make_timeblock_plot_frame(
    solution_df,
    value_col='n_new',
    route_filter=None,
    sort_routes_numeric=True,
):
    if value_col not in ['n_new', 'delta_n']:
        raise ValueError("value_col must be 'n_new' or 'delta_n'")

    df_plot = solution_df.copy()

    if route_filter is not None:
        df_plot = df_plot[df_plot['route'].isin(route_filter)].copy()

    if route_filter is not None:
        route_order = list(route_filter)
    else:
        route_vals = df_plot['route'].unique().tolist()
        route_order = sort_route_index(route_vals) if sort_routes_numeric else route_vals

    timeblock_order = [tb for tb in TIME_BLOCKS if tb in df_plot['time_block'].unique()]

    plot_df = (
        df_plot[['route', 'time_block', 'n_old', 'n_new', 'delta_n']]
        .copy()
    )
    plot_df['route'] = pd.Categorical(plot_df['route'], categories=route_order, ordered=True)
    plot_df['time_block'] = pd.Categorical(plot_df['time_block'], categories=timeblock_order, ordered=True)
    plot_df = plot_df.sort_values(['route', 'time_block']).reset_index(drop=True)
    plot_df['plot_value'] = plot_df[value_col]

    return plot_df, route_order, timeblock_order


def make_plot_labels(time_block='all', value_col='n_new'):
    block_label = 'all time blocks' if time_block == 'all' else time_block
    y_label = {
        'n_new': 'Bus allocation (n_new)',
        'delta_n': 'Bus change from baseline (delta_n)',
    }[value_col]
    return block_label, y_label


def _style_single_bars(
    ax,
    plot_df,
    show_baseline_ticks=True,
    show_delta_labels=True,
    show_legend_baseline=True,
    color_bars_by_delta=True,
):
    baseline_handle_added = False

    for patch, (_, row) in zip(ax.patches, plot_df.iterrows()):
        x_left = patch.get_x()
        width = patch.get_width()
        x_mid = x_left + width / 2

        n_new_val = row['n_new']
        n_old_val = row['n_old']
        delta_val = row['delta_n']

        if color_bars_by_delta:
            if delta_val > 0:
                patch.set_facecolor('tab:green')
            elif delta_val < 0:
                patch.set_facecolor('tab:red')
            else:
                patch.set_facecolor('tab:gray')

        if show_baseline_ticks and pd.notna(n_old_val):
            tick_label = 'n_old baseline' if (show_legend_baseline and not baseline_handle_added) else '_nolegend_'
            ax.hlines(
                y=n_old_val,
                xmin=x_mid - width * 0.35,
                xmax=x_mid + width * 0.35,
                colors='black',
                linewidth=2,
                label=tick_label,
                zorder=4,
            )
            if show_legend_baseline and not baseline_handle_added:
                baseline_handle_added = True

        if show_delta_labels and pd.notna(delta_val) and int(round(delta_val)) != 0:
            y_text = n_new_val + 0.15
            va = 'bottom'
            if n_new_val < 0:
                y_text = n_new_val - 0.15
                va = 'top'
            ax.text(
                x_mid,
                y_text,
                f'{int(round(delta_val)):+d}',
                ha='center',
                va=va,
                fontsize=8,
                zorder=5,
            )


def _style_grouped_bars(
    ax,
    plot_df,
    show_baseline_ticks=True,
    show_delta_labels=True,
    show_legend_baseline=True,
):
    baseline_handle_added = False

    for patch, (_, row) in zip(ax.patches, plot_df.iterrows()):
        x_left = patch.get_x()
        width = patch.get_width()
        x_mid = x_left + width / 2

        n_new_val = row['n_new']
        n_old_val = row['n_old']
        delta_val = row['delta_n']

        if show_baseline_ticks and pd.notna(n_old_val):
            tick_label = 'n_old baseline' if (show_legend_baseline and not baseline_handle_added) else '_nolegend_'
            ax.hlines(
                y=n_old_val,
                xmin=x_mid - width * 0.35,
                xmax=x_mid + width * 0.35,
                colors='black',
                linewidth=2,
                label=tick_label,
                zorder=4,
            )
            if show_legend_baseline and not baseline_handle_added:
                baseline_handle_added = True

        if show_delta_labels and pd.notna(delta_val) and int(round(delta_val)) != 0:
            if n_new_val >= 0:
                y_text = n_new_val + 0.12
                va = 'bottom'
            else:
                y_text = n_new_val - 0.12
                va = 'top'
            ax.text(
                x_mid,
                y_text,
                f'{int(round(delta_val)):+d}',
                ha='center',
                va=va,
                fontsize=7,
                rotation=90,
                zorder=5,
            )


def plot_all_routes_bar(
    solution_df,
    time_block='all',
    value_col='n_new',
    figsize=(20, 8),
    sort_routes_numeric=True,
    show_baseline_ticks=True,
    show_delta_labels=True,
    show_legend_baseline=True,
    color_bars_by_delta=True,
):
    block_label, y_label = make_plot_labels(
        time_block=time_block,
        value_col=value_col,
    )

    if time_block == 'all':
        plot_df, route_order, timeblock_order = make_timeblock_plot_frame(
            solution_df=solution_df,
            value_col=value_col,
            route_filter=None,
            sort_routes_numeric=sort_routes_numeric,
        )

        pivot = plot_df.pivot(index='route', columns='time_block', values='plot_value').reindex(index=route_order, columns=timeblock_order)
        ax = pivot.plot(kind='bar', figsize=figsize)
        ax.set_title(f'All routes: {value_col} by route with time-block bars')
        ax.set_xlabel('Route')
        ax.set_ylabel(y_label)

        if value_col == 'delta_n':
            ax.axhline(0, linewidth=1)
        elif value_col == 'n_new':
            _style_grouped_bars(
                ax=ax,
                plot_df=plot_df,
                show_baseline_ticks=show_baseline_ticks,
                show_delta_labels=show_delta_labels,
                show_legend_baseline=show_legend_baseline,
            )
        plt.xticks(rotation=90)
        plt.tight_layout()
        plt.show()
        return

    plot_df = make_plot_frame(
        solution_df=solution_df,
        time_block=time_block,
        value_col=value_col,
        route_filter=None,
        sort_routes_numeric=sort_routes_numeric,
    )

    ax = plot_df.set_index('route')['plot_value'].plot(kind='bar', figsize=figsize)
    ax.set_title(f'All routes: {value_col} by route | {block_label}')
    ax.set_xlabel('Route')
    ax.set_ylabel(y_label)

    if value_col == 'delta_n':
        ax.axhline(0, linewidth=1)
    elif value_col == 'n_new':
        _style_single_bars(
            ax=ax,
            plot_df=plot_df,
            show_baseline_ticks=show_baseline_ticks,
            show_delta_labels=show_delta_labels,
            show_legend_baseline=show_legend_baseline,
            color_bars_by_delta=color_bars_by_delta,
        )
        if show_legend_baseline:
            ax.legend()
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()


def plot_selected_routes_bar(
    solution_df,
    routes_of_interest,
    time_block='all',
    value_col='n_new',
    figsize=(12, 6),
    show_baseline_ticks=True,
    show_delta_labels=True,
    show_legend_baseline=True,
    color_bars_by_delta=True,
):
    block_label, y_label = make_plot_labels(
        time_block=time_block,
        value_col=value_col,
    )

    if time_block == 'all':
        plot_df, route_order, timeblock_order = make_timeblock_plot_frame(
            solution_df=solution_df,
            value_col=value_col,
            route_filter=routes_of_interest,
            sort_routes_numeric=False,
        )

        pivot = plot_df.pivot(index='route', columns='time_block', values='plot_value').reindex(index=route_order, columns=timeblock_order)
        ax = pivot.plot(kind='bar', figsize=figsize)
        ax.set_title(f'Selected routes: {value_col} by route with time-block bars')
        ax.set_xlabel('Route')
        ax.set_ylabel(y_label)

        if value_col == 'delta_n':
            ax.axhline(0, linewidth=1)
        elif value_col == 'n_new':
            _style_grouped_bars(
                ax=ax,
                plot_df=plot_df,
                show_baseline_ticks=show_baseline_ticks,
                show_delta_labels=show_delta_labels,
                show_legend_baseline=show_legend_baseline,
            )
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()
        return

    plot_df = make_plot_frame(
        solution_df=solution_df,
        time_block=time_block,
        value_col=value_col,
        route_filter=routes_of_interest,
        sort_routes_numeric=False,
    )

    ax = plot_df.set_index('route')['plot_value'].plot(kind='bar', figsize=figsize)
    ax.set_title(f'Selected routes: {value_col} by route | {block_label}')
    ax.set_xlabel('Route')
    ax.set_ylabel(y_label)

    if value_col == 'delta_n':
        ax.axhline(0, linewidth=1)
    elif value_col == 'n_new':
        _style_single_bars(
            ax=ax,
            plot_df=plot_df,
            show_baseline_ticks=show_baseline_ticks,
            show_delta_labels=show_delta_labels,
            show_legend_baseline=show_legend_baseline,
            color_bars_by_delta=color_bars_by_delta,
        )
        if show_legend_baseline:
            ax.legend()

    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


#### Run optimization

In [ ]:
result = solve_scenario(
    df=df,
    scenario=BASE_SCENARIO_ACTIVE,
    model_name='oc_transpo_taylor_linearized_r12',
    output_flag=OPTIMIZATION_SETUP['output_flag'],
)

solution_results = result['solution_df'].copy()
summary_results = result['summary'].copy()

summary_results

#### Summary tables

In [ ]:
route_summary = make_route_summary(
    solution_df=solution_results,
    time_block=TABLE_SETUP['route_summary_time_block'],
)

route_summary_sorted = route_summary.sort_values(
    TABLE_SETUP['sort_route_summary_by'],
    ascending=TABLE_SETUP['ascending'],
).reset_index(drop=True)

timeblock_summary = make_timeblock_summary(solution_results)

display(route_summary_sorted.head(TABLE_SETUP['top_n_routes']))
display(timeblock_summary)

#### Selected-route detail

In [ ]:
selected_route_detail = make_selected_route_detail(
    solution_df=solution_results,
    routes_of_interest=ROUTES_OF_INTEREST,
)

display(selected_route_detail)

#### Plot outputs

In [ ]:
plot_all_routes_bar(
    solution_df=solution_results,
    time_block=GRAPH_SETUP['time_block'],
    value_col=GRAPH_SETUP['value_col'],
    sort_routes_numeric=GRAPH_SETUP['sort_routes_numeric'],
    show_baseline_ticks=GRAPH_SETUP['show_baseline_ticks'],
    show_delta_labels=GRAPH_SETUP['show_delta_labels'],
    show_legend_baseline=GRAPH_SETUP['show_legend_baseline'],
    color_bars_by_delta=GRAPH_SETUP['color_bars_by_delta'],
)

In [ ]:
plot_selected_routes_bar(
    solution_df=solution_results,
    routes_of_interest=ROUTES_OF_INTEREST,
    time_block=GRAPH_SETUP['time_block'],
    value_col=GRAPH_SETUP['value_col'],
    show_baseline_ticks=GRAPH_SETUP['show_baseline_ticks'],
    show_delta_labels=GRAPH_SETUP['show_delta_labels'],
    show_legend_baseline=GRAPH_SETUP['show_legend_baseline'],
    color_bars_by_delta=GRAPH_SETUP['color_bars_by_delta'],
)